<a href="https://colab.research.google.com/github/JorgeCaldeira/Beware-the-dog/blob/main/DL_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##TODO
Please add max 2 pages to the report of Milestone 0 with:

*   data pipeline, including preprocessing
baseline description:

    *  model name / architecture depth;
    *  input shape;
    *  loss;
    *  optimiser;
    *  training budget (epochs, LR, batch, GPU time)


* preliminary results and training curves
* known issues and next steps, including a detailed plan for the rest of the experiments






By the first milestone, it is expected that you have the data downloaded and ready to be used, the models clearly identified and a skeleton code to start working on.

This time, the document should be 3 pages max and it can include whatever is still relevant from the proposal. It should have:
* Title, group members
* Introduction: this section introduces your problem, and the overall plan for approaching your problem
* Problem statement: Describe your problem precisely specifying the dataset to be used, expected results and evaluation
* Technical Approach: Describe the methods you intend to apply to solve the given problem
* Figure describing the approach / methodology.

Get inspiration from the papers that you’re reading for your project. Some examples:
figure 1 of this paper or similar
check the posters here or go check LASIGE posters in C6 floor 2.
Describe the experiments you are planning to do and preliminary results if you have them.



In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pltr
from google.colab import drive
import os

In [8]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
for root, dirs, files in os.walk('/content/drive'):
    if 'dog-breed-identification' in root:
        print(root)

/content/drive/.shortcut-targets-by-id/1OBtF3Z4ZtdyduyQMzRNMvGbRBtun6fZc/AP Projeto/dog-breed-identification


KeyboardInterrupt: 

In [9]:
import os
os.chdir('/content/drive/.shortcut-targets-by-id/1OBtF3Z4ZtdyduyQMzRNMvGbRBtun6fZc/AP Projeto/dog-breed-identification')

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import torch.optim as optim

labels_df = pd.read_csv('labels.csv')
breed_list = sorted(labels_df['breed'].unique())
breed_to_idx = {breed: i for i, breed in enumerate(breed_list)}

class DogDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_id = self.dataframe.iloc[idx, 0]
        breed_name = self.dataframe.iloc[idx, 1]
        label = breed_to_idx[breed_name]

        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [11]:
imagenet_normalization = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

baseline_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_normalization
])

augmented_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    imagenet_normalization
])

In [13]:
def initialize_model():
    model = models.resnet50(weights='DEFAULT') # slides dizem para usarmos uma pre treinada ja (semana 4 acho eu)

    for param in model.parameters():
        param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 120)

    return model

dog_model = initialize_model().to(device)
criterion = nn.CrossEntropyLoss() # na semana 3 isto é mencionado
optimizer = optim.Adam(dog_model.fc.parameters(), lr=0.0001)

train_set = DogDataset(labels_df, 'train/', transform=augmented_transform)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_hits = 0
    total_samples = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        predictions = model(images)
        loss = criterion(predictions, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted_class = predictions.max(1)
        total_samples += targets.size(0)
        correct_hits += predicted_class.eq(targets).sum().item()

    return running_loss / total_samples, 100. * correct_hits / total_samples

num_epochs = 5
loss_history = []
acc_history = []

for epoch in range(num_epochs):
    epoch_loss, epoch_acc = train_epoch(dog_model, train_loader, criterion, optimizer, device)
    loss_history.append(epoch_loss)
    acc_history.append(epoch_acc)
    print(f"Epoch {epoch+1}/{num_epochs} Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(loss_history, marker='o', color='red')
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')

plt.subplot(1, 2, 2)
plt.plot(acc_history, marker='o', color='blue')
plt.title('Training Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.show()

KeyboardInterrupt: 